# Day 2: answering the question

**Python for Data Analysis**, Coding for Reproducible Research (CfRR), University of Exeter

Yesterday we cleaned every month of Oxford weather since 1853 and found that summer 2026 tops the ranking. Today we find out whether that means anything: we plot it, fit a trend and test how sure we can be of it, bring in a second station, combine and resample the two records, and make sure the whole analysis can be rerun by someone else.

## By the end of today you will be able to

- explain why a notebook must run top to bottom from a fresh kernel
- build a matplotlib figure step by step with `fig, ax`, and save it with the numbers behind it
- make interactive charts with Plotly, and say when each library is the better tool
- say what a NumPy array is, why vectorised code beats a loop, and how `NaN` behaves in NumPy and pandas
- fit a straight-line trend, say how sure you can be of it, and what it cannot tell you
- turn repeated steps into a function and check it with `assert`
- combine tables with `concat` and `merge`, reshape them from long to wide, and resample a time series to years or seasons

## How to use this notebook

As yesterday, there are two versions. In `day2.ipynb` most code cells are empty, and you type the code into them as the instructor types it on screen; the few cells that are filled in already, just run. [`day2_solutions.ipynb`](day2_solutions.ipynb), in the same folder, has every cell filled in, with its output: copy any cell you miss from there.

*Data: Met Office historic station data for Oxford and Lerwick. Contains public sector information licensed under the [Open Government Licence v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).*

## 1. Recap, and "Restart and run all"

*10:00, 10 minutes*

Yesterday's cleaning lives in `day2_checkpoint.ipynb`, in the same folder as this notebook. The first cell, filled in for you, runs it with `%run`, which hands us its variables, including the cleaned `oxford` DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

%run day2_checkpoint.ipynb
oxford.shape

The checkpoint also defined the file location and column names we will reuse later.

### The page is not the program

Here are yesterday's summer averages again, in two cells:

Now delete the `summer` cell and run the `summer_tmax` cell again. It still works. Then choose **Kernel > Restart Kernel and Run All Cells**, and it fails with `NameError: name 'summer' is not defined`.

The variable `summer` stayed in the kernel's memory after its cell was deleted, so everything kept working until the kernel restarted. A notebook remembers everything you have ever run, in the order you ran it, not the order on the page. The only reliable test is to restart and run everything from the top. Put the missing line back, above the cell that uses it.

## 2. Seeing it

*10:10, 45 minutes*

A matplotlib chart has two main objects: the **Figure** (the whole image) and the **Axes** (one plot inside it, with its own axes, labels and lines). Every chart here starts with `fig, ax = plt.subplots()` and then adds things to `ax`. Writing it this way takes a few more lines than a one-liner, but you control everything. The [CfRR Matplotlib page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Matplotlib.html) has many more chart types.

Start with every month in the record.

Unreadable: the seasonal cycle swamps everything else. Choose the chart for the question. Ours is about summers, so plot one value per summer.

(Why not the mean for each calendar year? 2026 only has nine months so far, all from the warmer part of the year, which would flatter it.)

Year-to-year noise hides any slow change. A 10-year rolling mean averages each summer with the nine before it. Add it as a second, thicker line, with a legend to tell them apart.

Because we kept hold of `ax`, we can keep adding to the same chart. Highlight 2026, then save the figure as a file you could put in a paper.

Save the numbers behind the chart as well as the picture. `.to_csv` writes a Series or DataFrame to a CSV file, which opens in anything from Python to Excel:

### The same thing with Plotly

*Last 15 minutes of this block*

Plotly makes interactive charts that you can hover over, zoom and pan. `plotly.express` builds a whole chart in one call. The [CfRR Plotly page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/data-analysis-plotly.html) goes further.

`px.line` takes a table and the names of its columns. The years are `summer_tmax`'s index, not a column, so `reset_index()` first turns them back into a column, `yyyy`.

Hover over the points to find the three hottest summers. Drag across the 1970s to zoom in, and double-click to zoom back out.

Plotly has a function for each kind of chart, all used the same way: a table, then the names of columns. A **box plot** shows how values are spread. The box runs from the 25% value to the 75% value, as in yesterday's `.describe()`, the line across it is the median, and dots beyond the whiskers are unusual months. One box for each month of the year:

Hover over the dot above the July box, then the dots above August: `hover_data` adds the year.

Which tool, when?

- **Plotly:** quick to write and good for exploring; charts can be shared as web pages.
- **matplotlib:** more lines of code, but control over every detail, and it saves the static image files that journals want. (Plotly can save static images too, but needs the extra `kaleido` package and a copy of Chrome.)

## 3. Is it a trend?

*10:55, 40 minutes*

To fit a trend we use NumPy, the library underneath pandas. First, a look at what it is.

### What's underneath: NumPy

*First 10 minutes of this block*

Every pandas column is a NumPy array with a name and an index attached. NumPy is what makes the arithmetic fast. The [CfRR NumPy page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_NumPy.html) has the full tour.

The checkpoint gave `oxford` a `tmean` column: the mean temperature for each month, the average of its mean maximum and mean minimum. Here is that calculation the way the Introduction to Python course would write it, with a loop:

NumPy can do the same calculation without a loop. Arithmetic on an array applies to every element at once: this is called **vectorisation**.

Same numbers, less code. It is also much faster. `%timeit` runs a line many times and reports how long it takes:

The vectorised version is many times faster, and the gap grows with the size of the data. That is why pandas and NumPy code avoids looping over rows.

pandas columns work the same way. The checkpoint made `tmean` with `(oxford["tmax"] + oxford["tmin"]) / 2`, which lines the values up by row, and yesterday's masks, such as `df["tmax"] > 20`, compared every month at once.

### How NaN behaves

December 1860 has no `tmin`. Here are three ways to average `tmin`, missing month included:

`np.mean` returns `nan`: any arithmetic with `NaN` gives `NaN`, so one missing month spoils the whole result. `np.nanmean` skips missing values. pandas skips them by default, as yesterday's `.count()` showed. Neither choice is wrong, but you need to know which one you are getting.

### Fitting a straight line

NumPy fits lines too. `np.polyfit(x, y, 1)` finds the straight line that best fits the points (degree 1 means a straight line). It returns the slope and the intercept. Like most NumPy functions, it takes arrays or pandas columns.

Positive: summers have been warming. The x values are years, so the slope is in °C per year, and times ten gives °C per decade.

To see the trend, draw it. We will build one chart through the rest of this section, adding each result to it as we go, so keep hold of `fig` and `ax`. A straight line's value in any year is `slope * year + intercept`, and, as with `(tmax + tmin) / 2`, one line of vectorised arithmetic works it out for every year in `summer_tmax.index` at once.

A single line through every summer since 1853 treats the whole record alike. Fit it again using only summers from 1970 onwards. `summer_tmax` is labelled by year, so `.loc` takes a range of years, as it took a range of dates yesterday: `summer_tmax.loc[1970:]` is every summer from 1970 to the end.

About five times steeper. Add it to the chart: as in block 2, because we kept hold of `ax` we can draw on the same chart again, and `fig` on the last line shows it.

A line through every summer since 1853 hides the fact that most of the warming is recent. The start year is a choice, and it changes the answer.

### How sure are we?

A few dozen noisy summers could slope upwards by chance. `scipy.stats.linregress` fits the same straight line as `np.polyfit`, and also says how far to trust it. SciPy, the scientific library built on NumPy, is one of the libraries you installed before the course.

Four numbers, in turn:

- **Slope:** 0.47 °C per decade, the same as `np.polyfit`.
- **Standard error** (`stderr`): how far off the slope could easily be. Roughly, the true rate is within two standard errors of it: here, somewhere between 0.25 and 0.69 °C per decade.
- **p-value:** if summers were not warming at all, how often would chance alone give a slope this steep? Less than once in 10,000, so chance alone is very unlikely to explain it.
- **r²:** the share of the ups and downs from one summer to the next that the line explains. Only a quarter: most of the difference between one summer and the next is weather, not the trend.

The standard error is easier to judge on the chart. A fitted line always passes through the middle of the data: the mean year and the mean temperature. Through that point, draw the shallowest and the steepest lines within two standard errors of the slope, and shade between them. `ax.fill_between(x, lower, upper)` shades the area between two lines; `color="C2"` is matplotlib's third default colour, the green of the 1970 line, and `alpha=0.3` makes the shading see-through.

Every line in the fan slopes upwards: that is what the tiny p-value says. The summers themselves scatter far more widely than the fan: that is the small r². We can be fairly sure of the trend and still say little about any one summer.

### Unusual for its time?

Summers are warming, so a record is less surprising than it used to be. A fairer question: how far is 2026 from what the recent trend would predict? This cell is filled in, because it has several new ideas at once. Read it a line at a time:

1. fit the line to 1970 to 2025, so 2026 cannot pull the line towards itself
2. use the line to predict 2026
3. measure the typical scatter of summers around the line
4. compare 2026's distance from the line with that scatter

In [ ]:
before = summer_tmax.loc[1970:2025]
trend_slope, trend_intercept = np.polyfit(before.index, before, 1)
expected_2026 = trend_slope * 2026 + trend_intercept
scatter = (before - (trend_slope * before.index + trend_intercept)).std()
summer_tmax[2026] - expected_2026, scatter

The same comparison on the chart: the 1970 to 2025 line carried on to 2026, dashed, with a grey band one typical scatter either side of it, shaded with `ax.fill_between`. `np.arange(1970, 2027)` makes an array of the years 1970 to 2026 to draw them over. The dotted red line, just `ax.plot` between two points, is the gap between 2026 and the trend. Last, `ax.set_xlim` zooms in on the years since 1960, and the legend, now too long to sit beside the data, moves outside: `bbox_to_anchor=(1, 1)` pins its top-left corner to the top right of the axes.

2026 is about two and a half times as far from the trend as a typical summer. That ratio, the distance from what was expected divided by the typical scatter, is called a **z-score**. If summers scatter round the line in the usual bell-shaped way, fewer than 1 in 100 would land 2.5 or more above it: unusual even allowing for the warming.

What a line can and cannot tell you:

- It **can** summarise the average rate of change over the years you chose.
- It **cannot** tell you why, or promise that the next ten years will follow it.
- 2026 is still **provisional**, and every 2026 temperature is estimated. The Met Office may revise them.

## Break

*11:35, 10 minutes*

## 4. A second station

*11:45, 35 minutes*

Is Oxford typical? Lerwick, in Shetland, is over 900 km further north (60.14°N, 1.18°W, 82 metres above sea level) and surrounded by sea. Its record starts in December 1930.

Yesterday's cleaning steps need to run again for a second file, so we turn them into a function, `load_station(path)`, that returns the cleaned DataFrame. You do not need to type it from scratch: copy the cleaning cell from `day2_checkpoint.ipynb`, add a `def` line, indent everything, call the DataFrame `df` and add `return df`.

Does the function reproduce yesterday's result exactly?

### Checking with `assert`

`assert` checks something you believe is true, and stops with an error if it is not. It is the simplest kind of test. A failing one looks like this:

Here are three things that should be true of any station we load:

1. no month appears twice
2. temperatures are within a plausible range for the UK
3. cleaning has not lost any values: the number of missing values in each column matches the raw file

`check_station` is filled in for you. The three `assert` lines are the part to read; the rest just gets the numbers to compare.

In [ ]:
def check_station(df, path):
    raw = pd.read_csv(path, sep=r"\s+", skiprows=7, names=COLUMNS, na_values="---")
    assert df.index.is_unique, "a month appears more than once"
    assert df["tmax"].between(-20, 40).all(), "tmax outside a plausible range"
    lost = df[MEASURES].isna().sum() - raw[MEASURES].isna().sum()
    assert (lost == 0).all(), f"cleaning lost values:\n{lost}"

check_station(oxford, OXFORD)

No output is good news for an `assert`: Oxford passes all three checks.

Now the second station. Its file is in the `data` folder too:

In [ ]:
# Snapshot of https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/lerwickdata.txt
LERWICK = DATA / "lerwickdata.txt"
LERWICK.exists()

Read the error from the bottom up: a sunshine value ending in `#` could not be read. The header of the raw file says what it means. At Lerwick, a `#` after a sunshine value means it came from an automatic sensor rather than the older recorder. Our function was written for Oxford, which never uses `#`.

A loud failure like this is a good thing: it stopped us before anything went wrong.

### The quick fix again

An AI assistant suggests a quick fix: swap `.astype(float)` for `pd.to_numeric(..., errors="coerce")`. Make that one change and run the checks:

The error goes away, but every sunshine value with a `#` silently becomes missing: years of Lerwick sunshine, gone. The third check catches it. Without the check, the code would have run and every sunshine result for Lerwick would have been wrong.

The proper fix follows yesterday's pattern: strip the `#` as well as the `*`, and keep the information in a `sun_auto` column, just as we kept the estimates.

### Stacking the stations

To compare the stations, stack them into one table with a `station` column. `pd.concat` puts tables on top of each other, and `.assign` adds a column with the same value in every row. The result is **long** data: one row per station per month. Long data suits `groupby`.

Now the summer mean max for both stations, on one `ax`. Was 2026 a record summer in Lerwick too?

No. 2026 is not even in Lerwick's top three summers. The heat of 2026 was a southern story, which is exactly why one station is not enough to talk about "the UK".

## 5. Combining and resampling

*12:20, 30 minutes*

`both` is long: the stations stacked, one row per station per month. To read the stations side by side, **wide** data is easier: one column per station. There are two ways to get there, `unstack` and `merge`.

### From long to wide

`summers` has two levels of labels in its index, station and year. `unstack` moves the `station` level into the columns. You will also meet `pivot`, which does the same job starting from ordinary columns, and `melt`, which goes the other way, from wide back to long.

Do the two stations have their hot summers in the same years? `.corr()` gives the **correlation** between each pair of columns: 1 means they rise and fall together exactly, 0 means no connection, and -1 means one rises as the other falls.

0.56: a hot summer in Oxford tends to be a warm one in Lerwick, but only loosely. A scatter plot shows what that looks like. Hover over the point furthest to the right:

2026 is far to the right, but only halfway up: the southern story again.

### Side by side by date: `merge`

`unstack` worked because both stations were already in one table. `merge` lines up two separate tables wherever their **key** columns match, here the year and the month. Columns with the same name in both tables get a suffix to tell them apart:

It starts in December 1930, not 1853. By default `merge` keeps only the months that are in both tables: an **inner** join. `how="left"` keeps every row of the left table, Oxford, and leaves Lerwick's columns missing before 1930. Count the rows each way:

Which rows survive is a decision, like `dropna` yesterday. With the stations side by side, one subtraction compares them month by month. How much warmer are Oxford's days than Lerwick's, at each time of year?

1.5 °C in December and January, but 8.1 °C in July. The sea round Shetland is slow to warm and slow to cool, so Lerwick's summers stay cool and its winters mild.

### Resampling: months into years and seasons

Yesterday, `groupby("yyyy")` needed a year column. With dates as the index, `resample` groups by time directly. `"YE"` means calendar years, each labelled by its last day. Total rainfall for each year:

2026 looks like a drought, but it has only nine months so far. `.count()` shows how many months went into each total:

A part-year gives a misleading total or average, which is why we compare summers rather than years. `resample` makes seasons too. `"QS-DEC"` cuts the year into quarters starting in December, so they are winter (December to February), spring, summer and autumn, each labelled by its first month. The coldest winters:

The coldest is the winter of 1962 to 1963, the Big Freeze, labelled by its December. The quarters labelled June are our summers, made a second way. Two routes to the same numbers is a good check:

## 6. Wrap: so, was summer 2026 unusual?

*12:50, 10 minutes*

This cell, filled in, pulls the numbers together:

In [ ]:
previous = summer_tmax.drop(2026)
print(f"Summer 2026 mean max: {summer_tmax[2026]:.1f} °C")
print(f"Previous record: {previous.max():.1f} °C in {previous.idxmax()}")
print(f"Above the 1970 to 2025 trend by {summer_tmax[2026] - expected_2026:.1f} °C")
print(f"Typical scatter around the trend: {scatter:.1f} °C")
print(f"z-score: {(summer_tmax[2026] - expected_2026) / scatter:.1f}")

So what would we write, in one sentence, and which caveats belong in it? One version:

> Summer 2026 was the hottest in Oxford's record, which goes back to 1853, by more than a degree, and well above what even the recent warming trend would predict; the figures are still provisional, and the analysis should be rerun on a later snapshot once the Met Office confirms them.

### Reproducibility checklist

- **Pinned data:** a dated snapshot of the Met Office files, kept with the notebooks; [`data/README.md`](data/README.md) records where each file came from and its checksum.
- **Locked libraries:** `uv.lock` records exact versions; `requirements.txt` is exported from it for pip users.
- **Fresh kernel:** the notebook runs top to bottom after a restart.
- **Flags kept:** estimated, provisional and automatic-sensor values are marked, not dropped.
- **Checks:** `assert` statements catch silent mistakes when the code meets new data.

### Demo: uv (optional)

uv is a fast replacement for `pip` and `venv`. In a copy of the course repository:

```bash
uv sync             # create .venv with exactly the versions in uv.lock
uv run jupyter lab  # start JupyterLab inside that environment
uv add seaborn      # add a library and update the lock file
```

If you prefer pip, `pip install -r requirements.txt` gives you the same versions.

### Where next

- [Introduction to Machine Learning](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/introduction_to_machine_learning.html), for models that learn from data
- [Software Development Best Practices](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/software_development_best_practices.html), for proper testing
- [Version Control](https://coding-for-reproducible-research.github.io/CfRR_Courses/programme_information/version_control.html)